# 31 · Cross-Condition Transfer and Invariance Test

This notebook asks whether the overlap phase found in different conditional regimes is really the **same invariant** or instead a mixture of condition-specific mechanisms.

Previous notebooks showed:
- zeta vs GUE has a stable overlap phase
- the phase survives several perturbations
- the signal decomposes into distributional and local-block components
- the phase is condition-dependent across entropy and unevenness regimes

Now we ask:

> If we train a boundary in one condition, does it transfer to another?

## Conditions

We test two paired condition systems:
- **entropy transfer**: low entropy ↔ high entropy
- **unevenness transfer**: low unevenness ↔ high unevenness

## Tasks

For each transfer direction, we compare:
- **zeta vs GUE**
- **zeta vs Poisson**

## Main outputs

- transfer matrices
- transfer phase-gap bars
- bootstrap error bars for within-condition vs cross-condition transfer
- transfer asymmetry summaries

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 2200
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        all_gaps.extend(np.diff(bulk_vals).tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window statistics and feature builder

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

def ratio_std(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.std(r))

def build_windows_and_features(gaps, feature_family="minimal", k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])

    stats = {
        "entropy": np.array([window_entropy(w) for w in windows_n]),
        "unevenness": np.array([unevenness(w) for w in windows_n]),
        "ratio_mean": np.array([ratio_mean(w) for w in windows_n]),
        "symmetry": np.array([reversal_symmetry_score(w) for w in windows_n]),
    }

    if feature_family == "minimal":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["ratio_mean"][i]]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "full":
        X = np.array([
            [stats["entropy"][i], stats["unevenness"][i], stats["symmetry"][i], stats["ratio_mean"][i], ratio_std(windows_n[i])]
            for i in range(len(windows_n))
        ], dtype=float)
    elif feature_family == "raw_window":
        X = windows_n.copy()
    else:
        raise ValueError(f"Unknown feature family: {feature_family}")

    return windows_n, stats, X

def apply_condition_mask(stats, condition_name):
    if condition_name == "low_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] <= thr
    if condition_name == "high_entropy":
        thr = np.median(stats["entropy"])
        return stats["entropy"] > thr
    if condition_name == "low_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] <= thr
    if condition_name == "high_unevenness":
        thr = np.median(stats["unevenness"])
        return stats["unevenness"] > thr
    raise ValueError(condition_name)

## RBF classifier and transfer evaluator

In [ ]:
def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    return (X_train - mean) / std, (X_test - mean) / std

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -40, 40)))

def fit_logistic_regression(X, y, lr=0.1, n_steps=2500, reg=1e-4):
    Xb = np.column_stack([np.ones(len(X)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(n_steps):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        grad[1:] += reg * w[1:]
        w -= lr * grad
    return w

def decision_function_logistic(X, w):
    Xb = np.column_stack([np.ones(len(X)), X])
    return Xb @ w

def predict_proba_logistic(X, w):
    return sigmoid(decision_function_logistic(X, w))

def choose_prototypes(X, n_proto=20):
    idx = np.linspace(0, len(X) - 1, n_proto).astype(int)
    return X[idx]

def estimate_rbf_gamma(X):
    D = np.linalg.norm(X[:, None, :] - X[None, :, :], axis=2)
    med = np.median(D[D > 0])
    if med <= 0:
        med = 1.0
    return 1.0 / (2.0 * med * med)

def rbf_features(X, prototypes, gamma):
    D2 = np.sum((X[:, None, :] - prototypes[None, :, :])**2, axis=2)
    return np.exp(-gamma * D2)

def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def overlap_coefficient_from_hist(a, b, bins=30):
    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    if hi <= lo:
        return 1.0
    hist_a, edges = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hist_b, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    dx = edges[1] - edges[0]
    return float(np.sum(np.minimum(hist_a, hist_b)) * dx)

def fit_rbf_boundary(X0_train, X1_train):
    m = min(len(X0_train), len(X1_train))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]

    X_train = np.vstack([X0_train, X1_train])
    y_train = np.array([0] * len(X0_train) + [1] * len(X1_train))

    Xtr_std, _ = standardize_train_test(X_train, X_train)
    prototypes = choose_prototypes(Xtr_std, n_proto=min(20, len(Xtr_std)))
    gamma = estimate_rbf_gamma(Xtr_std)
    R_train = rbf_features(Xtr_std, prototypes, gamma)
    w = fit_logistic_regression(R_train, y_train, lr=0.15, n_steps=3500, reg=1e-4)

    return {
        "mean": X_train.mean(axis=0),
        "std": np.where(X_train.std(axis=0) == 0, 1.0, X_train.std(axis=0)),
        "prototypes": prototypes,
        "gamma": gamma,
        "w": w,
    }

def evaluate_rbf_boundary(model, X0_test, X1_test):
    m = min(len(X0_test), len(X1_test))
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]

    X_test = np.vstack([X0_test, X1_test])
    y_test = np.array([0] * len(X0_test) + [1] * len(X1_test))

    Xte_std = (X_test - model["mean"]) / model["std"]
    R_test = rbf_features(Xte_std, model["prototypes"], model["gamma"])
    scores = decision_function_logistic(R_test, model["w"])
    preds = (predict_proba_logistic(R_test, model["w"]) >= 0.5).astype(int)

    acc = accuracy(y_test, preds)
    scores0 = scores[y_test == 0]
    scores1 = scores[y_test == 1]

    return {
        "accuracy": acc,
        "score_overlap": overlap_coefficient_from_hist(scores0, scores1, bins=30),
    }

def transfer_eval(X0_train, X1_train, X0_test, X1_test):
    model = fit_rbf_boundary(X0_train, X1_train)
    return evaluate_rbf_boundary(model, X0_test, X1_test)

## Bootstrap transfer helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_transfer(X0_train, X1_train, X0_test, X1_test, n_boot=30, seed=9423):
    boot_rng = np.random.default_rng(seed)
    acc_vals = []
    overlap_vals = []

    m = min(len(X0_train), len(X1_train), len(X0_test), len(X1_test))
    X0_train = X0_train[:m]
    X1_train = X1_train[:m]
    X0_test = X0_test[:m]
    X1_test = X1_test[:m]

    for _ in range(n_boot):
        A0 = bootstrap_rows(X0_train, boot_rng)
        A1 = bootstrap_rows(X1_train, boot_rng)
        B0 = bootstrap_rows(X0_test, boot_rng)
        B1 = bootstrap_rows(X1_test, boot_rng)

        out = transfer_eval(A0, A1, B0, B1)
        acc_vals.append(out["accuracy"])
        overlap_vals.append(out["score_overlap"])

    acc_vals = np.array(acc_vals)
    overlap_vals = np.array(overlap_vals)

    return {
        "acc_mean": float(acc_vals.mean()),
        "acc_std": float(acc_vals.std()),
        "acc_q025": float(np.quantile(acc_vals, 0.025)),
        "acc_q975": float(np.quantile(acc_vals, 0.975)),
        "overlap_mean": float(overlap_vals.mean()),
        "overlap_std": float(overlap_vals.std()),
        "overlap_q025": float(np.quantile(overlap_vals, 0.025)),
        "overlap_q975": float(np.quantile(overlap_vals, 0.975)),
    }

## Reduced experiment grid

In [ ]:
window_sizes = [3, 5, 7]
feature_family = "minimal"
sample_size = 140
height_block = (0, 400)
n_boot = 30

condition_pairs = {
    "entropy": ("low_entropy", "high_entropy"),
    "unevenness": ("low_unevenness", "high_unevenness"),
}

## Base gap slices

In [ ]:
start, stop = height_block
zeta_base_gaps = zeta_gaps_full[start:stop]
poisson_base_gaps = poisson_gaps_full[start:stop]
gue_base_gaps = gue_gaps_full[:max(stop - start + 300, 900)]
len(zeta_base_gaps), len(poisson_base_gaps), len(gue_base_gaps)

## Build condition-specific feature sets

In [ ]:
def get_conditioned_features(gaps, condition_name, k=5, feature_family="minimal", n=140):
    _, stats, X = build_windows_and_features(gaps, feature_family=feature_family, k=k)
    mask = apply_condition_mask(stats, condition_name)
    Xc = X[mask]
    return Xc[:min(len(Xc), n)]

conditioned = {}

for k in window_sizes:
    conditioned[k] = {}
    for cond_a, cond_b in condition_pairs.values():
        for condition_name in [cond_a, cond_b]:
            conditioned[k][("zeta", condition_name)] = get_conditioned_features(zeta_base_gaps, condition_name, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("poisson", condition_name)] = get_conditioned_features(poisson_base_gaps, condition_name, k=k, feature_family=feature_family, n=sample_size)
            conditioned[k][("gue", condition_name)] = get_conditioned_features(gue_base_gaps, condition_name, k=k, feature_family=feature_family, n=sample_size)

{k: {key: len(val) for key, val in conditioned[k].items()} for k in window_sizes}

## Main cross-condition transfer sweep

In [ ]:
transfer_results = []

for system_name, (cond_lo, cond_hi) in condition_pairs.items():
    for k in window_sizes:
        for train_cond, test_cond in [(cond_lo, cond_lo), (cond_lo, cond_hi), (cond_hi, cond_lo), (cond_hi, cond_hi)]:
            z_train = conditioned[k][("zeta", train_cond)]
            z_test = conditioned[k][("zeta", test_cond)]
            g_train = conditioned[k][("gue", train_cond)]
            g_test = conditioned[k][("gue", test_cond)]
            p_train = conditioned[k][("poisson", train_cond)]
            p_test = conditioned[k][("poisson", test_cond)]

            m = min(len(z_train), len(z_test), len(g_train), len(g_test), len(p_train), len(p_test), sample_size)
            if m < 40:
                continue

            out_zg = bootstrap_transfer(z_train[:m], g_train[:m], z_test[:m], g_test[:m], n_boot=n_boot, seed=10000 + 10*k + len(train_cond) + 100*len(test_cond))
            out_zp = bootstrap_transfer(z_train[:m], p_train[:m], z_test[:m], p_test[:m], n_boot=n_boot, seed=11000 + 10*k + len(train_cond) + 100*len(test_cond))

            transfer_results.append({
                "system": system_name,
                "k": k,
                "train_condition": train_cond,
                "test_condition": test_cond,
                "sample_used": m,
                "zg_acc_mean": out_zg["acc_mean"],
                "zg_acc_q025": out_zg["acc_q025"],
                "zg_acc_q975": out_zg["acc_q975"],
                "zg_overlap_mean": out_zg["overlap_mean"],
                "zg_overlap_q025": out_zg["overlap_q025"],
                "zg_overlap_q975": out_zg["overlap_q975"],
                "zp_acc_mean": out_zp["acc_mean"],
                "zp_acc_q025": out_zp["acc_q025"],
                "zp_acc_q975": out_zp["acc_q975"],
                "zp_overlap_mean": out_zp["overlap_mean"],
                "zp_overlap_q025": out_zp["overlap_q025"],
                "zp_overlap_q975": out_zp["overlap_q975"],
                "phase_gap_mean": out_zg["overlap_mean"] - out_zp["overlap_mean"],
            })

len(transfer_results)

## Quick diagnostic

In [ ]:
sorted(set(r["system"] for r in transfer_results)), sorted(set(r["train_condition"] for r in transfer_results)), sorted(set(r["test_condition"] for r in transfer_results))

## Transfer matrix helper

In [ ]:
def build_transfer_matrix(system_name, metric_name, k=5):
    cond_lo, cond_hi = condition_pairs[system_name]
    labels = [cond_lo, cond_hi]
    M = np.full((2, 2), np.nan)

    for i, train_cond in enumerate(labels):
        for j, test_cond in enumerate(labels):
            match = next(
                r for r in transfer_results
                if r["system"] == system_name and r["k"] == k
                and r["train_condition"] == train_cond and r["test_condition"] == test_cond
            )
            M[i, j] = match[metric_name]
    return labels, M

## Transfer matrices: zeta vs GUE overlap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    labels, M = build_transfer_matrix(system_name, "zg_overlap_mean", k=5)
    im = ax.imshow(M, vmin=0, vmax=1)
    ax.set_title(f"{system_name}: zeta vs GUE")
    ax.set_xticks(np.arange(2), labels, rotation=20)
    ax.set_yticks(np.arange(2), labels)
    ax.set_xlabel("test condition")
    ax.set_ylabel("train condition")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="transfer overlap mean")
plt.tight_layout()
plt.show()

## Transfer matrices: zeta vs Poisson overlap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    labels, M = build_transfer_matrix(system_name, "zp_overlap_mean", k=5)
    im = ax.imshow(M, vmin=0, vmax=1)
    ax.set_title(f"{system_name}: zeta vs Poisson")
    ax.set_xticks(np.arange(2), labels, rotation=20)
    ax.set_yticks(np.arange(2), labels)
    ax.set_xlabel("test condition")
    ax.set_ylabel("train condition")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="transfer overlap mean")
plt.tight_layout()
plt.show()

## Transfer matrices: phase gap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    labels, M = build_transfer_matrix(system_name, "phase_gap_mean", k=5)
    im = ax.imshow(M)
    ax.set_title(f"{system_name}: phase gap")
    ax.set_xticks(np.arange(2), labels, rotation=20)
    ax.set_yticks(np.arange(2), labels)
    ax.set_xlabel("test condition")
    ax.set_ylabel("train condition")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="phase gap")
plt.tight_layout()
plt.show()

## Transfer phase-gap bars across window size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    cond_lo, cond_hi = condition_pairs[system_name]
    directions = [(cond_lo, cond_lo), (cond_lo, cond_hi), (cond_hi, cond_lo), (cond_hi, cond_hi)]

    width = 0.18
    x = np.arange(len(window_sizes))

    for offset, (train_cond, test_cond) in zip([-1.5, -0.5, 0.5, 1.5], directions):
        vals = []
        for k in window_sizes:
            match = next(r for r in transfer_results if r["system"] == system_name and r["k"] == k and r["train_condition"] == train_cond and r["test_condition"] == test_cond)
            vals.append(match["phase_gap_mean"])
        ax.bar(x + offset * width, vals, width, label=f"{train_cond}→{test_cond}")

    ax.set_xticks(x, window_sizes)
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("phase gap")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Bootstrap error-bar comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

for ax, system_name in zip(axes, ["entropy", "unevenness"]):
    cond_lo, cond_hi = condition_pairs[system_name]
    directions = [(cond_lo, cond_lo), (cond_lo, cond_hi), (cond_hi, cond_lo), (cond_hi, cond_hi)]

    for train_cond, test_cond in directions:
        xs = []
        means = []
        lows = []
        highs = []
        for k in window_sizes:
            match = next(r for r in transfer_results if r["system"] == system_name and r["k"] == k and r["train_condition"] == train_cond and r["test_condition"] == test_cond)
            xs.append(k)
            means.append(match["phase_gap_mean"])
            low = match["zg_overlap_mean"] - match["zg_overlap_q025"]
            high = match["zg_overlap_q975"] - match["zg_overlap_mean"]
            lows.append(max(0.0, low))
            highs.append(max(0.0, high))
        ax.errorbar(xs, means, yerr=np.vstack([np.array(lows), np.array(highs)]), marker="o", capsize=4, label=f"{train_cond}→{test_cond}")
    ax.set_title(system_name)
    ax.set_xlabel("window size k")
    ax.set_ylabel("phase gap")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Transfer asymmetry summary

In [ ]:
asymmetry_rows = []

for system_name, (cond_lo, cond_hi) in condition_pairs.items():
    for k in window_sizes:
        forward = next(r for r in transfer_results if r["system"] == system_name and r["k"] == k and r["train_condition"] == cond_lo and r["test_condition"] == cond_hi)
        reverse = next(r for r in transfer_results if r["system"] == system_name and r["k"] == k and r["train_condition"] == cond_hi and r["test_condition"] == cond_lo)
        asymmetry_rows.append({
            "system": system_name,
            "k": k,
            "forward_minus_reverse_phase_gap": float(forward["phase_gap_mean"] - reverse["phase_gap_mean"]),
        })

asymmetry_rows

## Compact summary

In [ ]:
summary = {
    "window_sizes": window_sizes,
    "feature_family": feature_family,
    "sample_size_target": sample_size,
    "height_block": f"{height_block[0]+1}-{height_block[1]}",
    "n_boot": n_boot,
    "systems": list(condition_pairs.keys()),
    "asymmetry_rows": asymmetry_rows,
}
summary

## Notes

- Strong cross-condition transfer suggests the same invariant is expressed in multiple regimes.
- Weak cross-condition transfer suggests condition-specific mechanisms.
- Transfer asymmetry suggests one regime may be more general while the other is more specialized.

## Next directions

A natural next notebook could do one of these:

1. surrogate cross-condition transfer  
2. cross-feature-family transfer  
3. conditional confidence geometry under transfer  
4. train on one condition, test on mixed conditions